In [1]:
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.util import fetch_wasabi_file
from histpy import Histogram

from scoords import SpacecraftFrame

from astropy.time import Time
import astropy.units as u

SED_KEV_TO_ERG = u.keV.to(u.erg)
KEV_TO_MEV = u.keV.to(u.MeV)
from astropy.coordinates import SkyCoord, Galactic

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from threeML import *
from threeML.io.package_data import get_path_of_data_file
from threeML.io.logging import silence_console_log
from astromodels import Parameter
from threeML.minimizer.minimization import CannotComputeCovariance

from jupyterthemes import jtplot
jtplot.style(context="talk", fscale=1, ticks=True, grid=False)
set_threeML_style()
silence_warnings()

from scipy.integrate import quad

import matplotlib.ticker as mticker

from pathlib import Path

import os

%matplotlib inline

12:02:54 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=277251;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=513898;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=74578;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=523904;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#65\65]8;;\
                  will not be available.                                                                           

12:02:55 WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=161719;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=885526;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

12:02:55 INFO      Starting 3ML!                                                                     ]8;id=158313;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=586325;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#44\44]8;;\

         WARNING   WARNINGs here are NOT errors                                                      ]8;id=610733;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=908916;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#45\45]8;;\

...

         WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=484072;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=460711;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#126\126]8;;\
                  software installed and configured?                                                               

12:02:56 WARNING   No fermitools installed                                              ]8;id=249985;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=220323;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

         WARNING   Env. variable OMP_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=177539;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=829259;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable MKL_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=418417;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=495649;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable NUMEXPR_NUM_THREADS is not set. Please set it to 1 for optimal     ]8;id=87028;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=170111;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

In [2]:
data_path = Path("/Users/parshadkp/Software/COSI_Data/")

In [3]:
from cosipy.event_selection import GoodTimeInterval
from agn_cosi_fit_utils import open_spacecraft_history, scale_spacecraft_livetime

orientation_path = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
source_coord = SkyCoord(l=172.104, b=-51.934, frame="galactic", unit="deg")
fov_cut = 60 * u.deg

full_sc_orientation = open_spacecraft_history(orientation_path)
source_gti = GoodTimeInterval.from_pointing_cut(
    source_coord,
    full_sc_orientation,
    fov_cut,
    earth_occ=False,
)
sc_orientation = full_sc_orientation.apply_gti(source_gti)

print(f"NGC 1068 FOV cut: {fov_cut.to_value(u.deg):.0f} deg")
print(f"Selected livetime: {sc_orientation.cumulative_livetime().to_value(u.s):,.1f} s")


NGC 1068 FOV cut: 60 deg
Selected livetime: 2,546,115.0 s


In [4]:
dr = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"

In [5]:
multiplier_1068 = 8
exposure_1068 = multiplier_1068 * 3

# Scale count histograms and response livetime together; keep source flux intrinsic.
sc_orientation = scale_spacecraft_livetime(sc_orientation, multiplier_1068)


# NGC 1068 CPL + PL comparison


## Injected cutoff power law (thermal)


### NGC 1068 ($E_c=128$ keV)


In [6]:
K_inj = 3.1e-1 / u.cm / u.cm / u.s / u.keV
piv_inj = 1.0 * u.keV
xc_inj = 128.0 * u.keV
index_inj = -2.10

spectrum_inj_1068 = Cutoff_powerlaw()
spectrum_inj_1068.K.value = K_inj.value
spectrum_inj_1068.piv.value = piv_inj.value
spectrum_inj_1068.xc.value = xc_inj.value
spectrum_inj_1068.index.value = index_inj
spectrum_inj_1068.K.unit = K_inj.unit
spectrum_inj_1068.piv.unit = piv_inj.unit
spectrum_inj_1068.xc.unit = xc_inj.unit


def cutoff_powerlaw_k_at_pivot(shape, pivot_value):
    """Return the equivalent CPL K after changing its pivot."""
    return float(shape.K.value * (pivot_value / shape.piv.value) ** shape.index.value)


## Injected power-law tail (non-thermal)


In [36]:
norm_nt = 0.42
tail_pivot_keV = 200.0
tail_index = -2.8

K_inj_tail = (
    norm_nt * spectrum_inj_1068.evaluate_at(tail_pivot_keV)
    / u.cm / u.cm / u.s / u.keV
)

spectrum_inj_1068_PL = Powerlaw()
spectrum_inj_1068_PL.K.value = K_inj_tail.value
spectrum_inj_1068_PL.piv.value = tail_pivot_keV
spectrum_inj_1068_PL.index.value = tail_index
spectrum_inj_1068_PL.K.unit = K_inj_tail.unit
spectrum_inj_1068_PL.piv.unit = u.keV

spectrum_inj_1068_total = spectrum_inj_1068 + spectrum_inj_1068_PL

# The linked fit uses both component normalizations at a 200 keV pivot.
thermal_k_at_200 = cutoff_powerlaw_k_at_pivot(spectrum_inj_1068, tail_pivot_keV)
linking_ratio_1068 = spectrum_inj_1068_PL.K.value / thermal_k_at_200

print("Thermal CPL K at a 200 keV pivot:", thermal_k_at_200)
print("Injected power-law K at 200 keV:", spectrum_inj_1068_PL.K.value)
print("Linked K ratio:", linking_ratio_1068)
print(
    "Flux-density ratio at 200 keV:",
    spectrum_inj_1068_PL.evaluate_at(200) / spectrum_inj_1068.evaluate_at(200),
)

thermal_energy_flux, _ = quad(
    lambda energy: energy * spectrum_inj_1068.evaluate_at(energy),
    100.0,
    10000.0,
)
nonthermal_energy_flux, _ = quad(
    lambda energy: energy * spectrum_inj_1068_PL.evaluate_at(energy),
    100.0,
    10000.0,
)
print(
    "Non-thermal 0.1-10 MeV energy-flux fraction:",
    nonthermal_energy_flux / (thermal_energy_flux + nonthermal_energy_flux),
)


Thermal CPL K at a 200 keV pivot: 4.562456144556676e-06
Injected power-law K at 200 keV: 4.0166395973616155e-07
Linked K ratio: 0.08803678260346114
Flux-density ratio at 200 keV: 0.4200000000000001
Non-thermal 0.1-10 MeV energy-flux fraction: 0.36317938227868196


# Spectral fitting


In [37]:
source_file = (
    data_path
    / "AGN_Data/GammaRay/Paper_Models"
    / "NGC1068_ec_128_-2.8_DC4_COSI_cpl_pl_60_fovCut_normNT_0p42.hdf5"
)
background_file = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    "COSI/Software_Files/DC4_Files/Background/"
    "Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_"
    "NGC1068_60deg_fov_cut.hdf5"
)

for required_file in (source_file, background_file):
    if not required_file.exists():
        raise FileNotFoundError(required_file)

ngc1068_source_hist = Histogram.open(source_file) * multiplier_1068
bkg = Histogram.open(background_file) * multiplier_1068

# Collapse the background time axis and match the source histogram metadata.
bkg = bkg.project("Em", "Phi", "PsiChi")
ngc1068_source_hist.axes["Em"].axis_scale = bkg.axes["Em"].axis_scale
ngc1068_source_hist = ngc1068_source_hist.to(unit=bkg.unit, update=False)
ngc1068_data_hist = ngc1068_source_hist + bkg

print(f"Loaded source: {source_file.name}")
print(f"Loaded background: {background_file.name}")


Loaded source: NGC1068_ec_128_-2.8_DC4_COSI_cpl_pl_60_fovCut_normNT_0p42.hdf5
Loaded background: Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_NGC1068_60deg_fov_cut.hdf5


## Perform the COSI-only spectral fits


Fit the injected CPL+PL model and a nested CPL-only model for comparison.


In [38]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter

bkg_par = Parameter(
    "background_cosi",
    1,
    min_value=0,
    max_value=5,
    delta=0.05,
    desc="Background parameter for the NGC 1068 COSI fit",
)

cosi = COSIPlugin(
    "cosi",
    dr=dr,
    data=ngc1068_data_hist.project("Em", "Phi", "PsiChi"),
    bkg=bkg.project("Em", "Phi", "PsiChi"),
    sc_orientation=sc_orientation,
    nuisance_param=bkg_par,
    earth_occ=True,
)


### Thermal cutoff-power-law component


In [39]:
l = 172.104
b = -51.934

# Express the injected CPL at the 200 keV pivot used by the fit.
K = cutoff_powerlaw_k_at_pivot(spectrum_inj_1068, 200.0) / u.cm / u.cm / u.s / u.keV
piv = 200.0 * u.keV
xc = 128.0 * u.keV
index = -2.10

spectrum_cpl = Cutoff_powerlaw()
spectrum_cpl.K.value = K.value
spectrum_cpl.piv.value = piv.value
spectrum_cpl.xc.value = xc.value
spectrum_cpl.index.value = index
spectrum_cpl.index.fix = True

spectrum_cpl.K.min_value = 1e-8
spectrum_cpl.K.max_value = 1e-2
spectrum_cpl.xc.min_value = 10
spectrum_cpl.xc.max_value = 2000

spectrum_cpl.K.unit = K.unit
spectrum_cpl.piv.unit = piv.unit
spectrum_cpl.xc.unit = xc.unit


### Thermal + non-thermal model


In [40]:
K_tail = spectrum_inj_1068_PL.K.value / u.cm / u.cm / u.s / u.keV
piv_tail = 200.0 * u.keV
index_tail = -2.8

spectrum_pl = Powerlaw()
spectrum_pl.K.value = K_tail.value
spectrum_pl.piv.value = piv_tail.value
spectrum_pl.index.value = index_tail

spectrum_pl.K.min_value = 1e-10
spectrum_pl.K.max_value = 1e-4
spectrum_pl.index.min_value = -5
spectrum_pl.index.max_value = 1
spectrum_pl.index.delta = 0.25

spectrum_pl.K.unit = K_tail.unit
spectrum_pl.piv.unit = piv_tail.unit


In [41]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter

# Keep both components in one point source so response caching follows linked parameters.
ngc1068 = PointSource(
    "NGC1068",
    l=l,
    b=b,
    spectral_shape=spectrum_cpl + spectrum_pl,
)
model = Model(ngc1068)
cosi.set_model(model)


def make_cpl_only_source(name, reference_source):
    reference_shape = reference_source.spectrum.main.shape.functions[0]
    cpl_shape = Cutoff_powerlaw()

    for parameter_name in ("K", "piv", "xc", "index"):
        reference_parameter = getattr(reference_shape, parameter_name)
        cpl_parameter = getattr(cpl_shape, parameter_name)
        cpl_parameter.value = reference_parameter.value
        cpl_parameter.fix = reference_parameter.fix

        if reference_parameter.min_value is not None:
            cpl_parameter.min_value = reference_parameter.min_value
        if reference_parameter.max_value is not None:
            cpl_parameter.max_value = reference_parameter.max_value
        if reference_parameter.delta is not None:
            cpl_parameter.delta = reference_parameter.delta
        if reference_parameter.unit is not None:
            cpl_parameter.unit = reference_parameter.unit

    return PointSource(name, l=l, b=b, spectral_shape=cpl_shape)


ngc1068_cpl_only = make_cpl_only_source("NGC1068_cpl_only", ngc1068)
model_cpl_only = Model(ngc1068_cpl_only)

bkg_par_cpl_only = Parameter(
    "background_cosi_cpl_only",
    1,
    min_value=0,
    max_value=5,
    delta=0.05,
    desc="Background parameter for the CPL-only NGC 1068 fit",
)

cosi_cpl_only = COSIPlugin(
    "cosi_cpl_only",
    dr=dr,
    data=ngc1068_data_hist.project("Em", "Phi", "PsiChi"),
    bkg=bkg.project("Em", "Phi", "PsiChi"),
    sc_orientation=sc_orientation,
    nuisance_param=bkg_par_cpl_only,
    earth_occ=True,
)
cosi_cpl_only.set_model(model_cpl_only)


### COSI-only data lists


In [42]:
plugins = DataList(cosi)
plugins_cpl_only = DataList(cosi_cpl_only)


### CPL+PL versus CPL-only comparison


In [43]:
link_function = Line(a=0.0, b=linking_ratio_1068)
link_function.a.fix = True
link_function.b.min_value = 0.0

model.link(
    model["NGC1068"].spectrum.main.composite.K_2,
    model["NGC1068"].spectrum.main.composite.K_1,
    link_function,
)

like = JointLikelihood(model, plugins, verbose=False)
like_cpl_only = JointLikelihood(model_cpl_only, plugins_cpl_only, verbose=False)

result = like.fit()
result_cpl_only = like_cpl_only.fit()


12:10:51 INFO      set the minimizer to minuit                                             ]8;id=859498;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=615681;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

         INFO      set the minimizer to minuit                                             ]8;id=332022;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=110685;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

Best fit values:

,result,unit
parameter,,
NGC1068.spectrum.main.composite.K_1,(4.6 -2.1 +4) x 10^-6,1 / (keV s cm2)
NGC1068.spectrum.main.composite.xc_1,(1.28 -0.30 +0.4) x 10^2,keV
NGC1068.spectrum.main.composite.K_2.Line.b,(0.9 +/- 1.4) x 10^-1,
NGC1068.spectrum.main.composite.index_2,-2.8 +/- 0.9,
background_cosi,(2.48983 +/- 0.00015) x 10,Hz


Correlation matrix:

...

1.00,-0.94,0.07
-0.94,1.00,-0.30
0.07,-0.30,1.00


Values of -log(likelihood) at the minimum:

,-log(likelihood)
cosi_cpl_only,-3786219925.933871
total,-3786219925.933871


Values of statistical measures:

,statistical measures
AIC,-7572439845.867638
BIC,-7572439814.825025


In [44]:
from agn_cosi_fit_utils import COSIPlugin


def make_null_likelihood(data_hist):
    bkg_par_null = Parameter(
        "background_cosi_null",
        1,
        min_value=0,
        max_value=5,
        delta=0.05,
        desc="Background parameter for the NGC 1068 null fit",
    )
    cosi_null = COSIPlugin(
        "cosi_null",
        dr=dr,
        data=data_hist.project("Em", "Phi", "PsiChi"),
        bkg=bkg.project("Em", "Phi", "PsiChi"),
        sc_orientation=sc_orientation,
        nuisance_param=bkg_par_null,
        earth_occ=True,
    )

    spectrum_null = Powerlaw()
    spectrum_null.K.value = 1e-30
    spectrum_null.index.value = 1
    spectrum_null.K.fix = True
    spectrum_null.index.fix = True

    source_null = PointSource(
        "NGC1068_null",
        l=l,
        b=b,
        spectral_shape=spectrum_null,
    )
    model_null = Model(source_null)
    cosi_null.set_model(model_null)

    like_null = JointLikelihood(model_null, DataList(cosi_null), verbose=False)
    like_null.fit()
    return like_null


def get_likelihood_statistic(joint_likelihood):
    statistic = joint_likelihood.results.get_statistic_frame()["-log(likelihood)"]
    if "total" in statistic.index:
        return float(statistic.loc["total"])
    return float(statistic.sum())


def detection_ts(null_likelihood, source_likelihood):
    return 2.0 * (
        get_likelihood_statistic(null_likelihood)
        - get_likelihood_statistic(source_likelihood)
    )


def model_improvement_ts(reference_likelihood, test_likelihood):
    return 2.0 * (
        get_likelihood_statistic(reference_likelihood)
        - get_likelihood_statistic(test_likelihood)
    )


like_null = make_null_likelihood(ngc1068_data_hist)
TS_cpl = detection_ts(like_null, like_cpl_only)
TS_cpl_pl = detection_ts(like_null, like)
TS_cpl_pl_vs_cpl = model_improvement_ts(like_cpl_only, like)

fit_ts_comparison = pd.DataFrame(
    [
        {
            "spectrum": "NGC 1068 Ec=128 keV",
            "TS_CPL": TS_cpl,
            "TS_CPL_plus_PL": TS_cpl_pl,
            "Delta_TS_CPL_plus_PL_vs_CPL": TS_cpl_pl_vs_cpl,
            "Sigma_CPL": np.sqrt(max(TS_cpl, 0.0)),
            "Sigma_CPL_plus_PL": np.sqrt(max(TS_cpl_pl, 0.0)),
            "Sigma_added_PL": np.sqrt(max(TS_cpl_pl_vs_cpl, 0.0)),
        }
    ]
)
display(fit_ts_comparison)

TS = TS_cpl_pl


12:11:10 INFO      set the minimizer to minuit                                             ]8;id=108447;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=805638;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

Best fit values:

,result,unit
parameter,,
background_cosi_null,(2.49131 +/- 0.00011) x 10,Hz


Correlation matrix:

1.00


Values of -log(likelihood) at the minimum:

,-log(likelihood)
cosi_null,-3786219663.8776383
total,-3786219663.8776383


Values of statistical measures:

,statistical measures
AIC,-7572439325.7552595
BIC,-7572439315.407704


,spectrum,TS_CPL,TS_CPL_plus_PL,Delta_TS_CPL_plus_PL_vs_CPL,Sigma_CPL,Sigma_CPL_plus_PL,Sigma_added_PL
0,NGC 1068 Ec=128 keV,524.112465,533.2652,9.152735,22.893503,23.092536,3.025349
